In [ ]:
from pyspark.sql import functions as F
from pathlib import Path
import json
from datetime import datetime

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
EVIDENCIAS = BASE / "evidencias"
EVIDENCIAS.mkdir(exist_ok=True)

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Catálogo:", spark.conf.get("spark.sql.catalogImplementation"))

In [ ]:
TABLAS = [
    "bcrp_series_largas",
    "bcrp_json_rentabilidad",
    "sbs_afiliados",
    "sbs_rentabilidad",
    "sbs_valor_cuota",
    "memorias_indicadores",
    "sbs_indicadores_adicionales",
    "tabla_analitica_mensual"
]

conteos = {}

print("=== TABLAS REGISTRADAS ===")

spark.sql("SHOW TABLES IN afp_analytics").show(
    truncate=False
)

print("=== FILAS POR TABLA ===")

for nombre in TABLAS:
    cantidad = spark.table(
        f"afp_analytics.{nombre}"
    ).count()

    conteos[nombre] = cantidad
    print(f"{nombre}: {cantidad:,}")

print(
    "\nTOTAL:",
    f"{sum(conteos.values()):,}"
)

In [ ]:
CAMPOS_CRITICOS = {
    "bcrp_series_largas": [
        "codigo", "fecha_mes", "valor"
    ],
    "bcrp_json_rentabilidad": [
        "codigo", "fecha_mes", "valor"
    ],
    "sbs_afiliados": [
        "fecha_mes", "afp", "afiliados"
    ],
    "sbs_rentabilidad": [
        "fecha_mes", "afp", "fondo",
        "tipo_rentabilidad", "valor"
    ],
    "sbs_valor_cuota": [
        "fecha", "afp", "fondo",
        "valor_cuota"
    ],
    "memorias_indicadores": [
        "afp", "anio_ejercicio",
        "disponible_desde", "caracteres"
    ],
    "sbs_indicadores_adicionales": [
        "fecha", "indicador", "valor",
        "archivo_origen", "hoja_origen",
        "fila_excel", "columna_excel"
    ],
    "tabla_analitica_mensual": [
        "fecha_mes",
        "rent_real_integra_12m",
        "rent_real_habitat_12m",
        "rent_real_prima_12m",
        "rent_real_profuturo_12m",
        "ipc_lag1",
        "tipo_cambio_lag1",
        "afiliados_integra",
        "anio_ejercicio",
        "objetivo_bajo_rendimiento_mes_siguiente"
    ]
}

In [ ]:
resultados_completitud = []

for nombre, campos in CAMPOS_CRITICOS.items():

    df = spark.table(f"afp_analytics.{nombre}")
    total = conteos[nombre]

    campos_disponibles = [
        c for c in campos if c in df.columns
    ]

    faltantes_esquema = [
        c for c in campos if c not in df.columns
    ]

    if faltantes_esquema:
        raise ValueError(
            f"{nombre}: columnas inexistentes: "
            f"{faltantes_esquema}"
        )

    expresiones = [
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in campos_disponibles
    ]

    conteo_nulos = df.agg(*expresiones).first().asDict()

    for campo in campos_disponibles:

        nulos = int(conteo_nulos[campo] or 0)
        completos = total - nulos
        porcentaje = (
            completos / total * 100
            if total > 0 else None
        )

        resultados_completitud.append({
            "tabla": nombre,
            "campo": campo,
            "total": total,
            "completos": completos,
            "nulos": nulos,
            "completitud_pct": round(porcentaje, 2)
            if porcentaje is not None else None
        })

resumen_completitud = spark.createDataFrame(
    resultados_completitud
)

print("=== COMPLETITUD POR CAMPO CRÍTICO ===")

resumen_completitud.orderBy(
    "tabla", "campo"
).show(100, truncate=False)

In [ ]:
CLAVES = {
    "bcrp_series_largas": [
        "codigo", "fecha_mes"
    ],
    "bcrp_json_rentabilidad": [
        "codigo", "fecha_mes"
    ],
    "sbs_afiliados": [
        "fecha_mes", "afp"
    ],
    "sbs_rentabilidad": [
        "fecha_mes", "afp",
        "fondo", "tipo_rentabilidad"
    ],
    "sbs_valor_cuota": [
        "fecha", "afp", "fondo"
    ],
    "memorias_indicadores": [
        "afp", "anio_ejercicio",
        "parte", "archivo_original"
    ],
    "sbs_indicadores_adicionales": [
        "archivo_origen", "hoja_origen",
        "fila_excel", "columna_excel"
    ],
    "tabla_analitica_mensual": [
        "fecha_mes"
    ]
}

resultados_duplicados = []

print("=== UNICIDAD DE CLAVES ===")

for nombre, claves in CLAVES.items():

    df = spark.table(f"afp_analytics.{nombre}")

    faltantes = [
        c for c in claves if c not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"{nombre}: claves inexistentes: {faltantes}"
        )

    repetidos = (
        df.groupBy(*claves)
        .count()
        .filter(F.col("count") > 1)
    )

    grupos = repetidos.count()

    filas_excedentes = (
        repetidos.agg(
            F.sum(F.col("count") - 1).alias("n")
        ).first()["n"] or 0
    )

    resultados_duplicados.append({
        "tabla": nombre,
        "grupos_duplicados": grupos,
        "filas_duplicadas_excedentes": int(filas_excedentes)
    })

    print(
        nombre,
        "| grupos duplicados:", grupos,
        "| filas excedentes:", int(filas_excedentes)
    )

In [ ]:
REGLAS_RANGO = [
    (
        "sbs_afiliados",
        "afiliados",
        F.col("afiliados") < 0,
        "Afiliados negativos"
    ),
    (
        "sbs_valor_cuota",
        "valor_cuota",
        F.col("valor_cuota") <= 0,
        "Valor cuota no positivo"
    ),
    (
        "memorias_indicadores",
        "caracteres",
        F.col("caracteres") <= 0,
        "Texto de memoria vacío"
    ),
    (
        "tabla_analitica_mensual",
        "tipo_cambio_lag1",
        F.col("tipo_cambio_lag1") <= 0,
        "Tipo de cambio no positivo"
    )
]

resultados_rango = []

print("=== REGLAS ESTRICTAS DE RANGO ===")

for tabla, campo, condicion, descripcion in REGLAS_RANGO:

    df = spark.table(f"afp_analytics.{tabla}")

    cantidad = df.filter(condicion).count()

    resultados_rango.append({
        "tabla": tabla,
        "campo": campo,
        "regla": descripcion,
        "incumplimientos": cantidad
    })

    print(descripcion, ":", cantidad)

In [ ]:
from pyspark.sql.window import Window

final = spark.table(
    "afp_analytics.tabla_analitica_mensual"
)

print("=== COHERENCIA TEMPORAL ===")

# Memorias disponibles después del cierre del mes
memorias_futuras = final.filter(
    F.col("disponible_desde").isNotNull()
    & (
        F.col("disponible_desde")
        > F.last_day("fecha_mes")
    )
).count()

print("Memorias futuras:", memorias_futuras)

# Comprobar continuidad mensual
fechas = [
    r["fecha_mes"]
    for r in final.select("fecha_mes")
    .orderBy("fecha_mes")
    .collect()
]

from datetime import date

def mes_siguiente(fecha):
    anio = fecha.year + (fecha.month == 12)
    mes = 1 if fecha.month == 12 else fecha.month + 1
    return date(anio, mes, 1)

saltos = sum(
    mes_siguiente(a) != b
    for a, b in zip(fechas, fechas[1:])
)

print("Saltos en secuencia mensual:", saltos)

# Comparación de etiqueta contra brecha siguiente
ventana = Window.orderBy("fecha_mes")

auditoria = (
    final.select(
        "fecha_mes",
        "brecha_real_pp",
        "objetivo_bajo_rendimiento_mes_siguiente"
    )
    .withColumn(
        "brecha_siguiente",
        F.lead("brecha_real_pp").over(ventana)
    )
    .withColumn(
        "objetivo_esperado",
        F.when(
            F.col("brecha_siguiente").isNull(),
            F.lit(None).cast("int")
        ).otherwise(
            F.when(
                F.col("brecha_siguiente") < 0,
                1
            ).otherwise(0)
        )
    )
)

etiquetas_inconsistentes = auditoria.filter(
    ~F.col(
        "objetivo_bajo_rendimiento_mes_siguiente"
    ).eqNullSafe(
        F.col("objetivo_esperado")
    )
).count()

print("Etiquetas inconsistentes:", etiquetas_inconsistentes)

In [ ]:
print("=== REGISTROS QUE INCUMPLEN REGLAS CRÍTICAS ===")

condiciones = {
    "sbs_afiliados":
        F.col("afiliados").isNull()
        | (F.col("afiliados") < 0),

    "sbs_valor_cuota":
        F.col("valor_cuota").isNull()
        | (F.col("valor_cuota") <= 0),

    "memorias_indicadores":
        F.col("caracteres").isNull()
        | (F.col("caracteres") <= 0),

    "sbs_indicadores_adicionales":
        F.col("fecha").isNull()
        | F.col("indicador").isNull()
        | F.col("valor").isNull()
        | F.col("archivo_origen").isNull()
        | F.col("hoja_origen").isNull()
        | F.col("fila_excel").isNull()
        | F.col("columna_excel").isNull()
}

incumplimientos_criticos = {}

for tabla, condicion in condiciones.items():
    df = spark.table(f"afp_analytics.{tabla}")

    cantidad = df.filter(condicion).count()

    incumplimientos_criticos[tabla] = cantidad

    print(tabla, "| filas observadas:", cantidad)

In [ ]:
reporte_calidad = {
    "fecha": datetime.now().isoformat(),
    "filas_acumuladas": sum(conteos.values()),
    "tablas": len(TABLAS),
    "completitud": resultados_completitud,
    "duplicados": resultados_duplicados,
    "rangos": resultados_rango,
    "coherencia_temporal": {
        "memorias_futuras": memorias_futuras,
        "saltos_mensuales": saltos,
        "etiquetas_inconsistentes": etiquetas_inconsistentes
    },
    "incumplimientos_en_tablas_procesadas":
        incumplimientos_criticos,
    "nota_rechazos": (
        "No se dispone de un contador consolidado "
        "de registros rechazados durante todas las "
        "etapas históricas del ETL."
    )
}

ruta_calidad = (
    EVIDENCIAS / "16_reporte_calidad.json"
)

with ruta_calidad.open("w", encoding="utf-8") as f:
    json.dump(
        reporte_calidad,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Reporte guardado:", ruta_calidad)

In [ ]:
print("=== FORMATO Y RUTA DE LAS TABLAS ===")

for nombre in TABLAS:

    print("\nTABLA:", nombre)

    detalle = spark.sql(
        f"DESCRIBE EXTENDED afp_analytics.{nombre}"
    )

    detalle.filter(
        F.col("col_name").isin(
            "Provider",
            "Location",
            "Partition Provider"
        )
    ).show(truncate=False)

In [ ]:
print("=== PARTICIONES ANUALES ===")

for nombre in TABLAS:

    print("\nTABLA:", nombre)

    spark.sql(
        f"SHOW PARTITIONS afp_analytics.{nombre}"
    ).show(50, truncate=False)